# A1.2 — Les valeurs manquantes

1. **Analyse** : quelles colonnes manquent, ensemble ou séparément, et dans quelles bibliothèques.
2. **Reconstruction par règle** : une partie des NA de `collision_energy_ev` se recalcule exactement.
3. **Choix d'un imputeur par variable** : on masque des valeurs *observées*, on les prédit, et on compare l'erreur.
   Deux masquages : **au hasard** (20 % des valeurs observées) et **par bibliothèque** (on cache toute une
   bibliothèque, comme le font les vrais NA).

Tout est calculé sur les molécules du split `train` : la validation reste intacte pour l'étape suivante
(A4.1 : effet de chaque stratégie sur le MRR@25 de `dev`, sur toute la validation et sur timsTOF à part).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

from casmi.config import SEED
from casmi.io import load_meta, load_test
from casmi.missing import ce_summary, clean_fake_na, instrument_family, library_folds, mask_random, nce_to_ev
from casmi.split import load_split

pd.set_option("display.width", 200)

## 1. Analyse des NA

On convertit d'abord les faux NA de `instrument_type` (`"nan"`, `"N/A"`, chaîne vide) en vrais NA.

In [ ]:
meta = load_meta()
meta["instrument_type"] = clean_fake_na(meta["instrument_type"])
NA_COLS = ["base_peak_intensity", "collision_energy_ev", "collision_energy_orig", "instrument_type", "precursor_error_ppm"]
na = meta[NA_COLS].isna()
na.mean().mul(100).round(2).rename("% NA (train)")

### Les NA sont-ils simultanés ?

Chaque ligne est une combinaison de colonnes manquantes (True = NA), avec son nombre de spectres.

In [ ]:
na.value_counts().rename("spectres").to_frame()

### Le lien avec la bibliothèque

In [ ]:
by_lib = na.groupby(meta["ingest_lib"]).mean().mul(100).round(1)
by_lib["spectres"] = meta.groupby("ingest_lib").size()
by_lib.sort_values("spectres", ascending=False)

**Lecture.** `base_peak_intensity` est un NA « tout ou rien » par bibliothèque. `collision_energy_ev` manque
entièrement pour gnps, masaryk et drug_plus, et en partie ailleurs. Les spectres timsTOF (enveda-180) n'ont
aucun NA, comme le test visible :

In [ ]:
test = load_test()
test[["base_peak_intensity", "collision_energy_ev", "instrument_type"]].isna().mean().mul(100).rename("% NA (test visible)")

## 2. Reconstruction par règle : NCE → eV

Pour les Orbitrap, l'énergie est souvent donnée en NCE (énergie « normalisée »). Sur tous les spectres qui
ont les deux valeurs, `collision_energy_ev` vaut exactement `NCE × precursor_mz / 500`, arrondi au dixième.

In [ ]:
cs = ce_summary(meta["collision_energy_ev"])
rule = nce_to_ev(meta["collision_energy_orig"], meta["collision_energy_orig_units"], meta["precursor_mz"])
check = rule.notna() & (cs["n_ce"] == 1)
print(f"Spectres vérifiables : {check.sum():,}")
print(f"Règle exacte (à 0,05 eV près) : {((rule[check] - cs.loc[check, 'ce_mean']).abs() <= 0.051).mean():.2%}")
print(f"NA de collision_energy_ev reconstruits par la règle : {(rule.notna() & (cs['n_ce'] == 0)).sum():,} sur {(cs['n_ce'] == 0).sum():,}")

La règle ne reconstruit qu'environ 3 % des NA : pour les autres, le texte d'origine manque aussi
(gnps en entier, par exemple). Il faut un modèle d'imputation.

## 3. Choix d'un imputeur par variable

### Les variables d'entrée des imputeurs

Seulement des informations **présentes dans le test** : un imputeur doit pouvoir servir sur n'importe quel
spectre. `ingest_lib` est exclue : elle n'existe pas dans le test, et le masquage par bibliothèque n'aurait
plus de sens. Les NA restants dans les entrées sont gérés directement par les modèles.

In [ ]:
top_adducts = meta["adduct"].value_counts().index[:30]
X_all = pd.DataFrame({
    "precursor_mz": meta["precursor_mz"],
    "log_num_peaks": np.log10(meta["num_peaks"]),
    "ionization_mode": meta["ionization_mode"].astype("category"),
    "adduct": meta["adduct"].where(meta["adduct"].isin(top_adducts), "autre").astype("category"),
    "instrument": instrument_family(meta["instrument_type"]),
    "n_ce": cs["n_ce"].where(cs["n_ce"] > 0),
    "ce_mean": cs["ce_mean"],
    "ce_min": cs["ce_min"],
    "ce_max": cs["ce_max"],
    "log_bpi": np.log10(meta["base_peak_intensity"]),
})

split = load_split()
is_train = meta["inchikey14"].isin(split.loc[split["split"] == "train", "inchikey14"]).to_numpy()
X = X_all[is_train].reset_index(drop=True)
libs = meta.loc[is_train, "ingest_lib"].reset_index(drop=True)
print(f"Spectres du split train : {len(X):,}")

### Les imputeurs candidats et l'évaluation

Pour chaque variable, trois candidats, du plus simple au plus riche :
- **médiane (ou mode) globale** : une seule valeur pour tous ;
- **médiane (ou mode) par groupe** : par famille d'instrument et mode d'ionisation (par mode d'ionisation et adduct
  pour imputer l'instrument lui-même), avec repli sur la valeur globale ;
- **modèle conditionnel** : un gradient boosting (`HistGradientBoosting`) qui prédit la variable à partir des autres.

Pour chaque masque, on entraîne sur les valeurs observées non masquées, et on mesure l'erreur sur les valeurs masquées.

In [ ]:
MAX_FIT = 300_000  # sous-échantillon d'entraînement du modèle, pour la vitesse


def fit_predict(X, y, fit, pred, features, kind, group):
    """Prédictions des trois imputeurs sur les lignes pred, appris sur les lignes fit."""
    rng = np.random.default_rng(SEED)
    out = {}
    if kind == "num":
        glob = y[fit].median()
        out["médiane globale"] = np.full(pred.sum(), glob)
        stat = y[fit].groupby([X.loc[fit, g] for g in group], observed=True).median()
    else:
        glob = y[fit].mode()[0]
        out["mode global"] = np.full(pred.sum(), glob, dtype=object)
        stat = y[fit].groupby([X.loc[fit, g] for g in group], observed=True).agg(lambda s: s.mode()[0])
    keys = pd.MultiIndex.from_frame(X.loc[pred, group].astype("object"))
    out["par groupe"] = stat.reindex(keys).fillna(glob).to_numpy()

    idx = np.flatnonzero(fit)
    if len(idx) > MAX_FIT:
        idx = rng.choice(idx, MAX_FIT, replace=False)
    Model = HistGradientBoostingRegressor if kind == "num" else HistGradientBoostingClassifier
    model = Model(categorical_features="from_dtype", random_state=SEED)
    model.fit(X.loc[idx, features], y[idx])
    out["modèle"] = model.predict(X.loc[pred, features])
    return out


def evaluate(X, y, observed, features, kind, libs, group=("instrument", "ionization_mode")):
    """Erreur de chaque imputeur, en masquage aléatoire puis bibliothèque par bibliothèque."""
    masks = {"aléatoire (20 %)": mask_random(observed.to_numpy(), 0.2, SEED)}
    masks |= {f"bibliothèque : {lib}": m.to_numpy() for lib, m in library_folds(observed, libs).items()}
    rows = []
    for name, masked in masks.items():
        fit = observed.to_numpy() & ~masked
        preds = fit_predict(X, y, fit, masked, features, kind, list(group))
        truth = y[masked].to_numpy()
        for imputer, p in preds.items():
            score = np.mean(np.abs(p - truth)) if kind == "num" else np.mean(p == truth)
            rows.append({"masquage": name, "valeurs masquées": masked.sum(), "imputeur": imputer, "score": score})
    table = pd.DataFrame(rows).pivot_table(index=["masquage", "valeurs masquées"], columns="imputeur", values="score", sort=False)
    return table.round(3)

### `base_peak_intensity`

On prédit son logarithme en base 10 : l'intensité va de 1 à 9 milliards. Erreur = écart absolu moyen sur
`log10` (0,3 signifie un facteur 2 en moyenne). **Plus petit = meilleur.**

In [ ]:
features = [c for c in X.columns if c != "log_bpi"]
bpi_scores = evaluate(X, X["log_bpi"], X["log_bpi"].notna(), features, "num", libs)
bpi_scores

### `collision_energy_ev`

On prédit l'énergie moyenne du spectre (`ce_mean`, en eV). Les autres colonnes d'énergie sont retirées des
entrées, puisqu'elles manquent en même temps. Erreur = écart absolu moyen en eV. **Plus petit = meilleur.**

In [ ]:
features = [c for c in X.columns if c not in ("n_ce", "ce_mean", "ce_min", "ce_max")]
ce_scores = evaluate(X, X["ce_mean"], X["ce_mean"].notna(), features, "num", libs)
ce_scores

### `instrument_type` (famille d'instrument)

Erreur = taux de familles correctement retrouvées. **Plus grand = meilleur.**

In [ ]:
features = [c for c in X.columns if c != "instrument"]
y_instr = X["instrument"].astype("object")
instr_scores = evaluate(X, y_instr, y_instr.notna(), features, "cat", libs, group=("ionization_mode", "adduct"))
instr_scores

## 4. Conclusions

**Le masquage aléatoire est trop optimiste.** Pour `base_peak_intensity`, le modèle se trompe d'un facteur
4,7 en moyenne en masquage aléatoire (0,67 en `log10`), mais d'un facteur 100 environ quand on cache toute la
bibliothèque gnps (≈ 2,0) — pire qu'une simple médiane. C'est le masquage par bibliothèque qui reproduit les
vrais NA : les 7 bibliothèques sans `base_peak_intensity` ne l'ont jamais renseignée, et leurs appareils
diffèrent de ceux qui la renseignent.

| Variable | NA (train) | Imputeur retenu | Erreur (masquage par bibliothèque) | Commentaire |
|---|---|---|---|---|
| `base_peak_intensity` | 45,8 % | aucun fiable | ≈ facteur 100 sur gnps | imputer revient à inventer : à tester contre la suppression en A4.1 |
| `collision_energy_ev` | 13,3 % | **règle NCE → eV**, puis **modèle** | 15,3 eV en moyenne (médiane par groupe : 16,6 ; globale : 19,4) | la règle est exacte mais ne couvre que 11 064 NA |
| `instrument_type` | 1,4 % | **modèle** (familles) | de 49 % à 100 % de familles retrouvées selon la bibliothèque | NA rares ; autre option : une catégorie « Inconnu » |

**Stratégies à comparer en A4.1** (MRR@25 sur `dev`, puis sur toute la validation et sur timsTOF à part) :
- `base_peak_intensity` : supprimer la variable / l'imputer par le modèle avec une indicatrice « était manquant » ;
- `collision_energy_ev` et `instrument_type` : supprimer les spectres concernés de l'entraînement / imputer
  (règle puis modèle) avec une indicatrice.

Supprimer tous les spectres qui ont au moins un NA revient presque à n'entraîner que sur enveda-180 (timsTOF) :
il ne reste que 1 154 969 spectres complets sur 2 539 608.